# 09 · Image Anomaly Detection & PatchCore

## What this notebook covers
Industrial visual inspection — detecting defects, scratches, contamination in manufactured parts — is the canonical image anomaly detection problem. This notebook implements PatchCore, the current state-of-the-art method.

## PaDiM — the predecessor (narrative)
PaDiM (Patch Distribution Modelling, Defard et al. 2020) extracts patch-level features from a pretrained backbone and fits a multivariate Gaussian per spatial location. At test time, Mahalanobis distance from each patch to its spatial Gaussian is the anomaly score.

PaDiM was a significant step forward but has two limitations:
1. Fitting a full covariance matrix per patch position is memory-intensive
2. It doesn't capture correlations *across* spatial positions

**PatchCore** (Roth et al. 2022) addressed both by using a coreset of representative patch embeddings from the entire training set, rather than per-position Gaussians. It achieves AUROC>99% on MVTec AD while being faster at test time and using less memory.

## PatchCore: the algorithm
1. Extract patch-level features from a pretrained backbone (ResNet/WideResNet) at layer 2 and 3
2. Aggregate spatially adaptive neighbourhood-averaged features
3. Subsample to a coreset using greedy coreset selection (maximise minimum distance)
4. At test time: anomaly score = distance to nearest coreset embedding; anomaly map = per-patch distances

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.random_projection import SparseRandomProjection
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

# ── Synthetic "inspection" images: normal = grey textures, anomalies have blobs ──
def make_texture(n, size=64, normal=True):
    imgs = []
    for _ in range(n):
        img = np.random.randint(100, 160, (size, size, 3), dtype=np.uint8)
        # Add a regular grid pattern (normal texture)
        img[::8, :, :] = np.clip(img[::8, :, :] + 20, 0, 255)
        img[:, ::8, :] = np.clip(img[:, ::8, :] + 20, 0, 255)
        if not normal:
            # Inject a random dark blob (defect)
            cx, cy = np.random.randint(15, 50, 2)
            r = np.random.randint(4, 12)
            yy, xx = np.ogrid[:size, :size]
            mask = (xx - cx)**2 + (yy - cy)**2 <= r**2
            img[mask] = np.random.randint(0, 40)
        imgs.append(Image.fromarray(img))
    return imgs

train_imgs = make_texture(100, normal=True)
test_normal  = make_texture(40, normal=True)
test_anomaly = make_texture(40, normal=False)
print(f"Train: {len(train_imgs)}  |  Test normal: {len(test_normal)}  |  Test anomaly: {len(test_anomaly)}")

In [ ]:
transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ImageListDataset(Dataset):
    def __init__(self, images, transform):
        self.images = images; self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.transform(self.images[idx])

# Use WideResNet50 pretrained backbone
weights = Wide_ResNet50_2_Weights.DEFAULT
backbone = wide_resnet50_2(weights=weights)
backbone.eval()

# Hook into layer2 and layer3
features_cache = {}
def make_hook(name):
    def hook(module, input, output):
        features_cache[name] = output
    return hook
backbone.layer2.register_forward_hook(make_hook("layer2"))
backbone.layer3.register_forward_hook(make_hook("layer3"))

def extract_features(images, batch_size=16):
    loader = DataLoader(ImageListDataset(images, transform), batch_size=batch_size)
    all_feats = []
    with torch.no_grad():
        for batch in loader:
            backbone(batch)
            f2 = features_cache["layer2"]  # (B, C2, H2, W2)
            f3 = features_cache["layer3"]  # (B, C3, H3, W3)
            # Upsample f3 to f2 spatial size
            f3_up = nn.functional.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
            combined = torch.cat([f2, f3_up], dim=1)  # (B, C2+C3, H, W)
            B, C, H, W = combined.shape
            patches = combined.permute(0, 2, 3, 1).reshape(B * H * W, C)
            all_feats.append(patches.numpy())
    return np.concatenate(all_feats, axis=0)

print("Extracting training features...")
train_feats = extract_features(train_imgs)
print(f"Train feature shape: {train_feats.shape}")

In [ ]:
# ── Coreset subsampling (greedy k-centre) ─────────────────────────────────────
def greedy_coreset(X, coreset_size=500, seed=0):
    """Greedy coreset: iteratively add the point farthest from the current set."""
    np.random.seed(seed)
    rp = SparseRandomProjection(n_components=128, random_state=seed)
    X_proj = rp.fit_transform(X)
    n = len(X_proj)
    coreset_idx = [np.random.randint(0, n)]
    distances = np.full(n, np.inf)
    for _ in range(coreset_size - 1):
        new = X_proj[coreset_idx[-1]]
        d = np.linalg.norm(X_proj - new, axis=1)
        distances = np.minimum(distances, d)
        coreset_idx.append(int(np.argmax(distances)))
    return np.array(coreset_idx)

print("Building coreset...")
coreset_idx = greedy_coreset(train_feats, coreset_size=300)
memory_bank = train_feats[coreset_idx]
print(f"Coreset size: {memory_bank.shape}")

In [ ]:
# ── PatchCore scoring ──────────────────────────────────────────────────────────
def patchcore_score(images, memory_bank):
    """
    Returns (image_scores, patch_map) for a list of images.
    image_score = max patch distance = worst-case patch anomaly.
    """
    loader = DataLoader(ImageListDataset(images, transform), batch_size=8)
    scores = []
    with torch.no_grad():
        for batch in loader:
            backbone(batch)
            f2 = features_cache["layer2"]
            f3 = nn.functional.interpolate(features_cache["layer3"], size=f2.shape[-2:], mode="bilinear", align_corners=False)
            combined = torch.cat([f2, f3], dim=1)
            B, C, H, W = combined.shape
            for b_idx in range(B):
                patches = combined[b_idx].permute(1, 2, 0).reshape(H*W, C).numpy()
                # Distance to nearest coreset neighbour
                diff = patches[:, None, :] - memory_bank[None, :, :]
                dists = np.linalg.norm(diff, axis=2).min(axis=1)
                scores.append(dists.max())  # image-level score = worst patch
    return np.array(scores)

test_imgs_all = test_normal + test_anomaly
y_test = np.array([0]*40 + [1]*40)

print("Scoring test images...")
img_scores = patchcore_score(test_imgs_all, memory_bank)
auc = roc_auc_score(y_test, img_scores)
print(f"PatchCore image-level AUROC: {auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(img_scores[y_test==0], bins=20, alpha=0.7, color="steelblue", label="Normal", density=True)
axes[0].hist(img_scores[y_test==1], bins=20, alpha=0.7, color="tomato",    label="Anomaly", density=True)
threshold = np.percentile(img_scores[y_test==0], 95)
axes[0].axvline(threshold, color="black", linestyle="--", label=f"Threshold={threshold:.1f}")
axes[0].set_title(f"PatchCore score distribution (AUROC={auc:.3f})")
axes[0].legend(fontsize=8)

# Show example normal and anomaly images
examples = [(test_normal[0], "Normal"), (test_anomaly[0], "Anomaly")]
axes[1].axis("off")
for i, (img, label) in enumerate(examples):
    ax_new = fig.add_axes([0.55 + i*0.22, 0.1, 0.2, 0.8])
    ax_new.imshow(img)
    ax_new.set_title(label, fontsize=10)
    ax_new.axis("off")

plt.savefig(f"{base}/09_patchcore.png", dpi=120, bbox_inches="tight")
plt.show()